# 06 Job Structured Extraction

This notebook extracts structured information from an IT job posting.

Main goals:
- load a cleaned IT job posting from the processed dataset
- create a job advertisement text input
- use OpenAI `gpt-4o-mini` through LangChain
- define a structured Pydantic schema for job data
- extract required skills, nice-to-have skills, tools, responsibilities and requirements
- save the structured job posting as JSON

This step does not compare the job posting with a CV.

## 1. Imports and Settings

In [1]:
import os
import json
import pandas as pd

from pathlib import Path
from dotenv import load_dotenv
from typing import List, Optional
from pydantic import BaseModel, Field

from pymongo import MongoClient
import hashlib
import re
from datetime import datetime

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

In [2]:
load_dotenv(dotenv_path=Path("../.env"))
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [3]:
MONGODB_URI = os.getenv("MONGODB_URI")

In [4]:
client = MongoClient(MONGODB_URI)
db = client["cv_job_matching_agent"]
jobs_collection = db["analyzed_jobs"]
jobs_collection.create_index("job_key", unique=True)

'job_key_1'

In [5]:
PROCESSED_JOBS_DIR = Path("../data/processed/jobs")
OUTPUT_JOB_EXTRACTION_DIR = Path("../outputs/job_extraction")
DATABASE_DIR = Path("../database")

In [6]:
cleaned_jobs_path = PROCESSED_JOBS_DIR / "cleaned_it_job_postings.csv"
structured_job_output_path = OUTPUT_JOB_EXTRACTION_DIR / "structured_job.json"
database_path = DATABASE_DIR / "jobs.sqlite"

## 2. Load Cleaned IT Job Postings Dataset

In [7]:
df_jobs = pd.read_csv(cleaned_jobs_path)

In [8]:
df_jobs.shape

(6495, 20)

In [9]:
df_jobs.head()

,title,description,company_name,formatted_work_type,work_type,formatted_experience_level,listed_time,original_listed_time,expiry,job_posting_url,clean_job_title,clean_job_description,combined_text,title_has_it_keyword,tech_keyword_count,primary_it_job_category,all_it_job_categories,category_scores,matched_keywords,is_multi_category
0,Full Stack Engineer,"Location: Remote\nCompany Overview:SkillFit, a...",Ideando Inc,Full-time,FULL_TIME,Unknown,1.713493e+12,1.713493e+12,1.716085e+12,https://www.linkedin.com/jobs/view/2234533717/...,Full Stack Engineer,"Location: Remote Company Overview:SkillFit, a ...",full stack engineer location: remote company o...,False,17,Full Stack Development,Full Stack Development; Backend Development; F...,"{'Data Science / AI': 1, 'Data Engineering': 1...","{'Data Science / AI': ['python'], 'Data Engine...",True
1,Senior Developer,This individual will work with a high performa...,USLI,Full-time,FULL_TIME,Associate,1.713538e+12,1.713538e+12,1.716130e+12,https://www.linkedin.com/jobs/view/3475933396/...,Senior Developer,This individual will work with a high performa...,senior developer this individual will work wit...,False,9,Backend Development,Backend Development; Frontend Development; Ful...,"{'Data Analytics': 1, 'Data Engineering': 1, '...","{'Data Analytics': ['sql'], 'Data Engineering'...",True
2,Senior Software Engineer,"StyleAI is the AI-powered, all-in-one unified ...",StyleAI,Full-time,FULL_TIME,Unknown,1.713397e+12,1.713397e+12,1.715989e+12,https://www.linkedin.com/jobs/view/3586167732/...,Senior Software Engineer,"StyleAI is the AI-powered, all-in-one unified ...",senior software engineer styleai is the ai-pow...,True,10,Full Stack Development,Full Stack Development; Frontend Development; ...,"{'Data Engineering': 2, 'Frontend Development'...","{'Data Engineering': ['postgresql', 'kafka'], ...",True
3,DDI Engineer,Title: Infoblox/DNS EngineerLocation: 6860 Yos...,Xoriant,Contract,CONTRACT,Unknown,1.713277e+12,1.713277e+12,1.715869e+12,https://www.linkedin.com/jobs/view/3625991523/...,DDI Engineer,Title: Infoblox/DNS EngineerLocation: 6860 Yos...,ddi engineer title: infoblox/dns engineerlocat...,False,6,DevOps / Cloud,DevOps / Cloud; Cybersecurity; Backend Develop...,"{'Data Science / AI': 1, 'Backend Development'...","{'Data Science / AI': ['python'], 'Backend Dev...",True
4,Cybersecurity Test Engineer – Remote,Decision Point Security Inc. is currently seek...,"Decision Point Security, Inc.",Full-time,FULL_TIME,Unknown,1.712857e+12,1.712857e+12,1.728409e+12,https://www.linkedin.com/jobs/view/3710273988/...,Cybersecurity Test Engineer – Remote,Decision Point Security Inc. is currently seek...,cybersecurity test engineer – remote decision ...,False,6,DevOps / Cloud,DevOps / Cloud; Cybersecurity; Backend Develop...,"{'Backend Development': 1, 'DevOps / Cloud': 3...","{'Backend Development': ['.net'], 'DevOps / Cl...",True


## 3. Select One Job Posting for Extraction

In [10]:
job_row_index = 1
selected_job = df_jobs.iloc[job_row_index]

In [11]:
selected_job

title                                                          Senior Developer
description                   This individual will work with a high performa...
company_name                                                               USLI
formatted_work_type                                                   Full-time
work_type                                                             FULL_TIME
formatted_experience_level                                            Associate
listed_time                                                     1713538064000.0
original_listed_time                                            1713538064000.0
expiry                                                          1716130064000.0
job_posting_url               https://www.linkedin.com/jobs/view/3475933396/...
clean_job_title                                                Senior Developer
clean_job_description         This individual will work with a high performa...
combined_text                 senior dev

In [12]:
selected_job['description']

'This individual will work with a high performance team in the creation and maintenance of applications for our mobile, web and in-house systems. This individual will take on a work leadership role and has the interpersonal and technical skills required to guide a team. They displays poise and maturity in task completion and personal interactions and acknowledges their own responsibility and accountability. They utilize a deep understanding of Object Oriented principles in designing and developing solutions for new or existing systems. Understands, supports and provides guidance on the developer team fundamentals. This individual will communicate effectively with the business, other developers and with Information Technology leadership. They have excellent oral and written communication skills, customer service skills and the ability/flexibility to respond to and thrive in a fast-paced, ever-changing collaborative team environment. This individual has the ability to resolve team confli

In [13]:
title = selected_job["title"]
description = selected_job["description"]

In [14]:
job_text = f"""
Job title:
{title}

Job description:
{description}
"""

## 4. Create Job Text Hash

In [15]:
def normalize_text_for_hash(text):
    if text is None:
        return ""

    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)

    return text

In [16]:
def create_job_key(job_text):
    normalized_text = normalize_text_for_hash(job_text)

    text_hash = hashlib.sha256(
        normalized_text.encode("utf-8")
    ).hexdigest()

    return f"text_hash_{text_hash}"

In [17]:
job_key = create_job_key(job_text)

In [18]:
job_key

'text_hash_e5e31e9634a7bdd428551b3012d5dd28ee3558142f843a5819622159d8b89fbd'

## 5. Check Whether Job Posting Already Exists

In [19]:
existing_job = jobs_collection.find_one(
    {
        "job_key": job_key
    }
)


In [20]:
job_already_exists = existing_job is not None

In [21]:
job_already_exists

False

## 6. Define Structured Job Schema

In [22]:
class ExperienceRequirement(BaseModel):
    minimum_years: Optional[str] = Field(
        default=None,
        description="Minimum years of experience required for this specific requirement, if explicitly mentioned."
    )

    seniority_level: Optional[str] = Field(
        default=None,
        description="Seniority level related to this requirement, such as Junior, Mid, Medior, Senior, Lead, if visible or clearly inferable."
    )

    related_area_or_skill: Optional[str] = Field(
        default=None,
        description="Technology, role, domain or skill area related to this experience requirement, if mentioned."
    )

    experience_description: str = Field(
        description="One specific experience requirement explicitly mentioned in the job posting."
    )

    is_required: Optional[bool] = Field(
        default=None,
        description="True if this experience is required, False if it is preferred or nice-to-have, None if unclear."
    )

In [23]:
class EducationRequirement(BaseModel):
    degree_required: Optional[str] = Field(
        default=None,
        description="Required or preferred degree or education level, if explicitly mentioned."
    )

    field_of_study: Optional[str] = Field(
        default=None,
        description="Required or preferred field of study, if mentioned."
    )

    education_description: str = Field(
        description="One specific education requirement explicitly mentioned in the job posting."
    )

    is_required: Optional[bool] = Field(
        default=None,
        description="True if this education requirement is required, False if it is preferred or optional, None if unclear."
    )

In [24]:
class StructuredJobPosting(BaseModel):
    job_title: Optional[str] = Field(
        default=None,
        description="Job title extracted from the posting."
    )

    company_name: Optional[str] = Field(
        default=None,
        description="Company name, if visible in the job posting."
    )

    location: Optional[str] = Field(
        default=None,
        description="Job location, if visible."
    )

    work_mode: Optional[str] = Field(
        default=None,
        description="Work mode such as remote, hybrid, on-site or unknown."
    )

    employment_type: Optional[str] = Field(
        default=None,
        description="Employment type such as full-time, part-time, contract, internship or unknown."
    )

    job_category: Optional[str] = Field(
        default=None,
        description="Broad IT job category, such as Frontend Development, Backend Development, Data Analytics, DevOps, QA, Cybersecurity, etc."
    )

    required_skills: List[str] = Field(
        default_factory=list,
        description="Skills that are clearly required in the job posting."
    )

    nice_to_have_skills: List[str] = Field(
        default_factory=list,
        description="Skills that are listed as preferred, optional or nice-to-have."
    )

    programming_languages: List[str] = Field(
        default_factory=list,
        description="Programming languages mentioned in the job posting."
    )

    frameworks_and_libraries: List[str] = Field(
        default_factory=list,
        description="Frameworks and libraries mentioned in the job posting."
    )

    databases: List[str] = Field(
        default_factory=list,
        description="Databases mentioned in the job posting."
    )

    cloud_and_devops_tools: List[str] = Field(
        default_factory=list,
        description="Cloud platforms, DevOps tools and infrastructure tools mentioned in the job posting."
    )

    data_and_ai_tools: List[str] = Field(
        default_factory=list,
        description="Data analysis, BI, machine learning or AI tools mentioned in the job posting."
    )

    testing_tools: List[str] = Field(
        default_factory=list,
        description="Testing or QA tools mentioned in the job posting."
    )

    other_tools: List[str] = Field(
        default_factory=list,
        description="Other software tools mentioned in the job posting."
    )

    responsibilities: List[str] = Field(
        default_factory=list,
        description="Main responsibilities and tasks described in the job posting."
    )

    experience_requirements: List[ExperienceRequirement] = Field(
        default_factory=list,
        description=(
            "List of experience and seniority requirements explicitly mentioned in the job posting. "
            "Each item should represent one separate requirement, such as total years of experience, experience with a specific technology, previous role experience, domain experience or seniority level."
        )
    )
    
    education_requirements: List[EducationRequirement] = Field(
        default_factory=list,
        description=(
            "List of education requirements explicitly mentioned in the job posting. "
            "Each item should represent one separate education-related requirement, such as degree level, field of study, formal education, or equivalent practical experience."
        )
    )

    certifications: List[str] = Field(
        default_factory=list,
        description="Required or preferred certifications mentioned in the posting."
    )

    language_requirements: List[str] = Field(
        default_factory=list,
        description="Human language requirements mentioned in the job posting."
    )

    soft_skills: List[str] = Field(
        default_factory=list,
        description="Soft skills mentioned in the job posting."
    )

    unclear_or_missing_information: List[str] = Field(
        default_factory=list,
        description=(
            "Important job posting information that is missing, unclear or not explicitly stated, based only on the information needed for CV-job matching. "
            "Check whether the posting clearly states: job title, company name, location, work mode, employment type, seniority or experience level, required technical skills, nice-to-have skills, main responsibilities, education requirements, certifications and language requirements. "
            "Do not list irrelevant missing information."
        )
    )

## 7. Initialize OpenAI Model

In [25]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

In [26]:
structured_llm = llm.with_structured_output(StructuredJobPosting)

## 8. Create Job Structured Extraction Prompt

In [27]:
job_extraction_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are an AI assistant specialized in structured extraction from IT job postings.

Your task is to extract structured information from the provided IT job advertisement text.

General rules:
- Extract only information that is explicitly present in the job posting.
- Do not invent skills, tools, experience, education, certifications, responsibilities or company information.
- If information is not clearly mentioned, return null or an empty list.
- Do not return the string "None", "N/A" or "Unknown" for missing values.
- Avoid duplicate values inside the same list.
- Keep extracted values concise and useful for later CV-job matching.

Required and nice-to-have skills:
- Separate required skills from nice-to-have skills whenever possible.
- Classify a skill as required if the posting uses words such as "required", "must have", "need", "minimum", "strong experience", "proficiency", or if the skill is listed under requirements.
- Classify a skill as nice-to-have if the posting uses words such as "preferred", "nice to have", "plus", "advantage", "bonus", or "familiarity with".
- If it is unclear whether a skill is required or optional, classify it as required only if the text strongly suggests that it is necessary for the role.

Field interpretation:
- employment_type means the contract or engagement type, such as Full-time, Part-time, Contract, Freelance, Internship or Temporary.
- work_mode means the working arrangement, such as Remote, Hybrid, On-site, or a clearly stated arrangement such as "On-site 75% of the time".
- Do not confuse employment_type with work_mode.
- job_category should be a broad IT category, such as Software Development, Frontend Development, Backend Development, Full Stack Development, Data Analytics, Data Science / AI, DevOps / Cloud, QA / Testing, Cybersecurity, or IT Support / Administration.

Technology extraction:
- Focus on IT-related information: programming languages, frameworks, libraries, databases, cloud platforms, DevOps tools, testing tools, data tools, AI tools and software tools.
- Normalize common technology names when possible:
  - JS -> JavaScript
  - TS -> TypeScript
  - GCP -> Google Cloud Platform
  - Postgres -> PostgreSQL
  - HTML 5 -> HTML5
  - jSON -> JSON
- Do not place the same technology in many unrelated categories. For example, React should be a framework/library, Python should be a programming language, and PostgreSQL should be a database.

Experience and education:
- Extract experience requirements only if the posting mentions years of experience, seniority, previous role experience, or experience in a specific area.
- If seniority level is not clearly stated, return null.
- Extract education requirements only if the posting mentions degree, formal education, technical certification, or equivalent practical experience.
- If education is not mentioned, leave education_requirements as an empty list.

Unclear or missing information:
- Add an item to unclear_or_missing_information only if the information is truly missing or unclear and useful for later CV-job matching.
- Before adding a missing-information note, check whether the same information was already extracted in another field.
- Do not say that employment type is missing if employment_type has a value.
- Do not say that work mode is missing if work_mode has a value.
- Do not say that location is missing if location has a value.
- Do not say that certifications are missing if certifications has at least one item.
- Do not say that language requirements are missing if language_requirements has at least one item.
- Do not add generic missing notes that contradict extracted fields.

Task boundaries:
- Do not compare the job posting with a CV.
- Do not generate candidate recommendations.
- Do not evaluate candidate fit.

Return the result using the required structured schema.
"""
        ),
        (
            "human",
            """
Extract structured information from the following IT job posting:

{job_text}
"""
        )
    ]
)

## 9. Run Structured Job Extraction

In [28]:
current_time = datetime.now().isoformat(timespec="seconds")

if job_already_exists:
    jobs_collection.update_one(
        {"job_key": job_key},
        {
            "$inc": {"submission_count": 1},
            "$set": {"last_seen_at": current_time}
        }
    )

    structured_job_dict = existing_job["structured_job"]

else:
    job_extraction_chain = job_extraction_prompt | structured_llm

    structured_job_result = job_extraction_chain.invoke(
        {"job_text": job_text}
    )

    if hasattr(structured_job_result, "model_dump"):
        structured_job_dict = structured_job_result.model_dump()
    else:
        structured_job_dict = structured_job_result.dict()

    structured_job_dict["metadata"] = {
        "job_key": job_key,
        "model": "gpt-4o-mini",
        "extraction_type": "structured_job_extraction",
        "source": "cleaned_dataset_demo",
        "source_file": cleaned_jobs_path.name,
        "source_row_index": int(job_row_index)
    }

    job_document = {
        "job_key": job_key,
        "created_at": current_time,
        "first_seen_at": current_time,
        "last_seen_at": current_time,
        "submission_count": 1,
        "source": "cleaned_dataset_demo",
        "original_job_text": job_text,
        "structured_job": structured_job_dict
    }

    jobs_collection.insert_one(job_document)


## 10. Convert Structured Result to Dictionary

In [29]:
if "structured_job_dict" not in globals():
    raise ValueError(
        "structured_job_dict was not created. "
        "Please run the previous cell with MongoDB check and LLM extraction logic."
    )

In [30]:
if hasattr(structured_job_dict, "model_dump"):
    structured_job_dict = structured_job_dict.model_dump()
elif hasattr(structured_job_dict, "dict"):
    structured_job_dict = structured_job_dict.dict()
elif isinstance(structured_job_dict, dict):
    structured_job_dict = structured_job_dict
else:
    raise TypeError(
        f"Unsupported structured_job_dict type: {type(structured_job_dict)}"
    )

In [31]:
structured_job_dict

{'job_title': 'Senior Developer',
 'company_name': 'USLI',
 'location': None,
 'work_mode': 'On-site 75% of the time',
 'employment_type': 'Full-time',
 'job_category': 'Software Development',
 'required_skills': ['ASP.NET',
  'C#',
  'MVC',
  'Object Oriented Design/Development',
  'VB.NET',
  'Agile',
  'SCRUM',
  'JavaScript',
  'jQuery',
  'JSON',
  'Knockout.js',
  'Angular.js',
  'HTML5',
  'SQL Server',
  'SQL/XML'],
 'nice_to_have_skills': ['Emerging Microsoft technologies'],
 'programming_languages': ['C#', 'VB.NET', 'JavaScript'],
 'frameworks_and_libraries': ['ASP.NET',
  'MVC',
  'jQuery',
  'Knockout.js',
  'Angular.js',
  'HTML5'],
 'databases': ['SQL Server'],
 'cloud_and_devops_tools': [],
 'data_and_ai_tools': [],
 'testing_tools': [],
 'other_tools': [],
 'responsibilities': ['Day to day oversight and direction on medium size team initiatives',
  'Management and taking advantage of code re-use',
  'Designing and implementing new feature sets',
  'Responsible for the c

## 11. Preview Extracted Job Profile

In [32]:
job_profile = {
    "job_title": structured_job_dict.get("job_title"),
    "company_name": structured_job_dict.get("company_name"),
    "location": structured_job_dict.get("location"),
    "work_mode": structured_job_dict.get("work_mode"),
    "employment_type": structured_job_dict.get("employment_type"),
    "job_category": structured_job_dict.get("job_category"),
}

In [33]:
job_profile

{'job_title': 'Senior Developer',
 'company_name': 'USLI',
 'location': None,
 'work_mode': 'On-site 75% of the time',
 'employment_type': 'Full-time',
 'job_category': 'Software Development'}

## 12. Preview Extracted Skills and Tools

In [34]:
skills_preview = {
    "required_skills": structured_job_dict.get("required_skills", []),
    "nice_to_have_skills": structured_job_dict.get("nice_to_have_skills", []),
    "programming_languages": structured_job_dict.get("programming_languages", []),
    "frameworks_and_libraries": structured_job_dict.get("frameworks_and_libraries", []),
    "databases": structured_job_dict.get("databases", []),
    "cloud_and_devops_tools": structured_job_dict.get("cloud_and_devops_tools", []),
    "data_and_ai_tools": structured_job_dict.get("data_and_ai_tools", []),
    "testing_tools": structured_job_dict.get("testing_tools", []),
    "other_tools": structured_job_dict.get("other_tools", []),
}

In [35]:
skills_preview

{'required_skills': ['ASP.NET',
  'C#',
  'MVC',
  'Object Oriented Design/Development',
  'VB.NET',
  'Agile',
  'SCRUM',
  'JavaScript',
  'jQuery',
  'JSON',
  'Knockout.js',
  'Angular.js',
  'HTML5',
  'SQL Server',
  'SQL/XML'],
 'nice_to_have_skills': ['Emerging Microsoft technologies'],
 'programming_languages': ['C#', 'VB.NET', 'JavaScript'],
 'frameworks_and_libraries': ['ASP.NET',
  'MVC',
  'jQuery',
  'Knockout.js',
  'Angular.js',
  'HTML5'],
 'databases': ['SQL Server'],
 'cloud_and_devops_tools': [],
 'data_and_ai_tools': [],
 'testing_tools': [],
 'other_tools': []}

## 13. Convert Extracted Skills to DataFrame

In [36]:
skill_rows = []

skill_categories = [
    "required_skills",
    "nice_to_have_skills",
    "programming_languages",
    "frameworks_and_libraries",
    "databases",
    "cloud_and_devops_tools",
    "data_and_ai_tools",
    "testing_tools",
    "other_tools",
    "soft_skills",
    "language_requirements"
]
for category in skill_categories:
    for item in structured_job_dict.get(category, []):
        skill_rows.append(
            {
                "category": category,
                "item": item
            }
        )

job_skills_df = pd.DataFrame(skill_rows)


In [37]:
job_skills_df

,category,item
0,required_skills,ASP.NET
1,required_skills,C#
2,required_skills,MVC
3,required_skills,Object Oriented Design/Development
4,required_skills,VB.NET
5,required_skills,Agile
6,required_skills,SCRUM
7,required_skills,JavaScript
8,required_skills,jQuery
9,required_skills,JSON


## 14. Preview

In [38]:
responsibilities_df = pd.DataFrame({
    "responsibility": structured_job_dict.get("responsibilities", [])
})

In [39]:
responsibilities_df

,responsibility
0,Day to day oversight and direction on medium s...
1,Management and taking advantage of code re-use
2,Designing and implementing new feature sets
3,Responsible for the coding standards of the de...
4,Assists in recommending refactoring phases
5,Keep up with new technologies and new developm...
6,Mentor and support other team members on compl...


In [40]:
structured_job_dict.get("experience_requirements", {})

[{'minimum_years': '10+ years',
  'seniority_level': 'Senior',
  'related_area_or_skill': 'software development',
  'experience_description': 'minimum of 10+ years of software development experience',
  'is_required': True}]

In [41]:
structured_job_dict.get("education_requirements", {})

[{'degree_required': 'College degree/technical certifications',
  'field_of_study': None,
  'education_description': 'College degree/technical certifications or equivalent industry/technical experience',
  'is_required': True}]

## 15. Save Structured Job JSON

In [42]:
with open(structured_job_output_path, "w", encoding="utf-8") as file:
    json.dump(structured_job_dict, file, indent=4, ensure_ascii=False)